### How to generate a well sampled gaussian pulse

In [ ]:
import SpecialFunctions: erf, erfc
import LinearAlgebra: diag

#using KadanoffBaym
include("../../KadanoffBaym.jl/src/KadanoffBaym.jl")

using Plots

Consider a gaussian pulse
$$\xi(t) = \left(\frac{\Omega^2}{2 \pi}\right)^{1/4}\exp{\left(-\frac{\Omega^2}{4}(t-t_c)^2\right)}$$

In [ ]:
# Parameters
const tc = 0.5 # peak arrival time
const Ω = 80.0 # frequency bandwidth of the pulse

function ξ(t)
    return (Ω^2 / (2π))^(1/4) * exp(-(t - tc)^2 * (Ω^2 / 4))
end

Consider a modulator cavity given by the triple $(S_M, L_M, H_M)$

for which $L_M = \lambda(t) a$ where $\lambda(t) \triangleq \frac{\xi(t)}{\sqrt{\int_t^\infty \textrm{d}s |\xi(s)|^2}}$

In [ ]:
λ(t::Float64) = ξ(t) / sqrt(0.5 * erfc((t - tc) * Ω / sqrt(2)))
# γ(t::Float64) = ξ(t) / sqrt(0.5(1+ erf((t - tc) * Ω / sqrt(2))))
γ(t::Float64) = ξ(t)

# Hamiltonian
h(t::Float64) = ComplexF64(0.1 - 0.5im * (λ(t) + γ(t)))

# Initial condition
n_0 = 0.0

In [ ]:
Lesser0 = KadanoffBaym.GreenFunction(zeros(ComplexF64, 1, 1, 1, 1), KadanoffBaym.Lesser)
Greater0 = KadanoffBaym.GreenFunction(zeros(ComplexF64, 1, 1, 1, 1), KadanoffBaym.Greater)

Lesser0[1,1,1,1] = -1.0im * n_0
Greater0[1,1,1,1] = -1.0im * (n_0 + 1)

# Pack them in an ArrayPartition
u0 = KadanoffBaym.ArrayPartition(Lesser0, Greater0);

# Define commutator
⋊(a,b) = a * b - b * adjoint(a)

# Remember that `u` here is also an ArrayPartition-like element
function f(u, p, t1, t2)
    t = times[t1]
    GL, GG = u.x[1], u.x[2]
    
    rhsL = -1.0im * ((h(t) + 1.0im * γ(t)) * GL[t1,t2])
    rhsG = -1.0im * ((h(t) + 1.0im * γ(t)) * GG[t1,t2])
  return KadanoffBaym.ArrayPartition(rhsL, rhsG)
end

function f_diag(u, p, t1, t2)
    t = times[t1]
    GL, GG = u.x[1], u.x[2]
    
    rhsL = -1.0im * (h(t) ⋊ GL[t1,t2] + 1.0im * γ(t) * (GL[t1,t2] + GG[t1,t2]))
    rhsG = -1.0im * (h(t) ⋊ GG[t1,t2] + 1.0im * λ(t) * (GG[t1,t2] - GL[t1,t2]) + 2.0im * γ(t) * GG[t1,t2])
  return KadanoffBaym.ArrayPartition(rhsL, rhsG)
end

In [ ]:
# ODE problem is defined by the rhs, initial condition and time span
tspan = (0.0,1.0)
prob = KadanoffBaym.ODEProblem(f, u0, tspan, f_diag=f_diag)

# Algorithm to timestep is the (Kadanoff-Baym) ABM43 (only really this one exists)
alg = KadanoffBaym.KB{KadanoffBaym.ABM43}()

# This algorithm requires a fixed time
dt = 0.003

times = collect(first(tspan):dt:last(tspan)+dt)

@assert sum((ξ.(times).^2)) * dt ≈ 1
sol = KadanoffBaym.solve(prob, alg, dt);

Plotting

In [ ]:
times = first.(sol.t[:,1])
plot(times, ξ.(times))
plot!(times, λ.(times))
plot!(times, γ.(times))

In [ ]:
plot(times, -sol.u.x[1].data[:,:,1,1] |> diag .|> imag, label="Lesser")
plot!(times, -sol.u.x[2].data[:,:,1,1] |> diag .|> imag , label="Greater")
plot!(twinx(), times, ξ.(times), color=:green, legend=:right, label="Pulse", line=:dot)

In [ ]:
heatmap(-sol.u.x[1].data[:,:,1,1] .|> imag)

In [ ]:
heatmap(-sol.u.x[1].data[:,:,1,1] .|> real)